# Demo: flujo de delegación del orquestador multi-agente

Este notebook ejecuta una consulta que obliga al **Supervisor** a delegar en el **Investigador**, pasar los datos al **Analista**, sintetizar la respuesta y validarla antes de terminar.

Requiere el archivo `.env` con la API key configurada (ver README).

In [ ]:
from IPython.display import Image, Markdown, display

from config import LLM_MODEL, LLM_PROVIDER, verificar_credenciales
from graph import construir_grafo
from main import CONSULTA_DEMO, ejecutar, guardar_traza

verificar_credenciales()
print(f"Proveedor: {LLM_PROVIDER} | Modelo: {LLM_MODEL}")

## 1. Topología del grafo

El diagrama se genera directamente desde el grafo compilado con `get_graph().draw_mermaid_png()`. Las flechas punteadas son aristas condicionales: las decide el Supervisor (o el Validador) en tiempo de ejecución.

In [ ]:
grafo = construir_grafo()
try:
    display(Image(grafo.get_graph().draw_mermaid_png()))
except Exception:
    display(Markdown("```mermaid\n" + grafo.get_graph().draw_mermaid() + "\n```"))

## 2. Ejecución de la consulta

Cada bloque muestra un paso del grafo: la decisión del Supervisor con su evaluación según la rúbrica, las herramientas que usó cada especialista y el resultado del Validador.

In [ ]:
print(CONSULTA_DEMO)

In [ ]:
estado, registro = await ejecutar(CONSULTA_DEMO, grafo)

## 3. Quién aportó qué

El campo `contribuciones` del estado compartido registra cada aporte con su agente, iteración, instrucción recibida y herramientas usadas.

In [ ]:
for aporte in estado["contribuciones"]:
    herramientas = ", ".join(aporte["herramientas"]) or "ninguna"
    print(f"Iteración {aporte['iteracion']} | {aporte['agente']:<12} | herramientas: {herramientas}")
    print(f"    instrucción: {aporte['instruccion'][:150]}")

## 4. Decisiones del Supervisor y validación

In [ ]:
for decision in estado["decisiones"]:
    marca = " (forzada)" if decision["forzada"] else ""
    print(f"Iteración {decision['iteracion']} → {decision['siguiente']}{marca}")
    print(f"    evaluación: {decision['evaluacion_ultimo_aporte'][:200]}")

for validacion in estado["validaciones"]:
    print(f"Validación intento {validacion['intento']}: {'aprobada' if validacion['aprobada'] else 'rechazada'}")
    for problema in validacion["problemas"]:
        print(f"    - {problema}")

## 5. Respuesta final

In [ ]:
display(Markdown(estado["respuesta_final"]))

In [ ]:
guardar_traza(CONSULTA_DEMO, estado, registro, nombre="ejecucion_notebook")